# LLM Inference Engine from Scratch

We build and benchmark every major optimization real serving stacks use:

| # | Technique | Key win |
|---|-----------|----------|
| 1 | **Baseline** | Reference: no caching, O(n²) compute per token |
| 2 | **KV Cache** | Pre-fill once, decode O(1) per step; prefix reuse |
| 3 | **Continuous batching** | Fill GPU at every step, not every batch |
| 4 | **Speculative decoding** | k tokens per target forward pass instead of 1 |
| 5 | **Quantization** | INT8 and 4-bit: smaller model, faster matmuls |

**Model:** GPT-2 (117 M, free on HuggingFace). Speculative uses GPT-2 → GPT-2-medium.  
**Hardware:** Colab free T4 GPU.

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
!git clone https://github.com/YOUR_USERNAME/inference-engine.git 2>/dev/null || echo 'already cloned'
%cd inference-engine
!pip install -q -r requirements.txt

In [ ]:
import torch, psutil, os

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')
if device == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    props  = torch.cuda.get_device_properties(0)
    print(f'VRAM   : {props.total_memory / 1e9:.1f} GB')
print(f'RAM    : {psutil.virtual_memory().total / 1e9:.1f} GB')
print(f'PyTorch: {torch.__version__}')

---
## 1. Baseline — Naive Autoregressive Decoding

Every decode step passes the **full sequence** through the model.  
Each attention head recomputes keys and values for all previous tokens from scratch.

Cost of generating T tokens from a prompt of length N:  
$$\text{FLOPs} \propto \sum_{t=0}^{T} (N+t)^2 \approx O\left(T \cdot N^2\right)$$

This is the floor. Everything else beats it.

In [ ]:
from engine.baseline import BaselineEngine

PROMPT = (
    "The key insight behind modern large language model serving is that "
    "attention computation can be cached across decode steps. Explain in detail"
)
N_TOKENS = 80

baseline_eng = BaselineEngine()
text, baseline_stats = baseline_eng.generate(PROMPT, max_new_tokens=N_TOKENS)

print(f"Output   : {text[:200]}")
print(f"TTFT     : {baseline_stats['ttft_ms']:.1f} ms")
print(f"Latency  : {baseline_stats['latency_ms']:.1f} ms")
print(f"Throughput: {baseline_stats['throughput_tps']:.1f} tok/s")

---
## 2. KV Cache + Prefix Reuse

**Pre-fill phase:** run the full prompt through the model once; store the
Key and Value tensors produced at each transformer layer.  
**Decode phase:** each new token only attends to the *stored* K/V — O(1) extra
attention work per step regardless of how long the context has grown.

**Prefix reuse:** many requests share a common system prompt.  
We store its KV once; subsequent requests skip the pre-fill for that prefix entirely.

```
Without prefix reuse:  pre-fill(system_prompt + user_msg)
With prefix reuse:     lookup(system_prompt_kv) + pre-fill(user_msg only)
```

In [ ]:
from engine.kv_cache import KVCacheEngine

kv_eng = KVCacheEngine()

# ── Without prefix reuse ─────────────────────────────────────────────────────
_, kv_stats = kv_eng.generate(PROMPT, max_new_tokens=N_TOKENS)
print('── KV cache (no prefix reuse) ──')
print(f"TTFT      : {kv_stats['ttft_ms']:.1f} ms  (speedup vs baseline: {baseline_stats['ttft_ms']/kv_stats['ttft_ms']:.1f}x)")
print(f"Throughput : {kv_stats['throughput_tps']:.1f} tok/s  (vs {baseline_stats['throughput_tps']:.1f})")

# ── With prefix reuse ────────────────────────────────────────────────────────
SHARED = "The key insight behind modern large language model serving is that "
SUFFIX = "attention computation can be cached across decode steps. Explain in detail"

kv_eng.cache_prefix(SHARED)   # pre-compute KV for shared prefix
_, pfx_stats = kv_eng.generate(SHARED + SUFFIX, max_new_tokens=N_TOKENS, shared_prefix=SHARED)
print('\n── KV cache + prefix reuse ──')
print(f"TTFT      : {pfx_stats['ttft_ms']:.1f} ms  (speedup vs no-reuse: {kv_stats['ttft_ms']/pfx_stats['ttft_ms']:.1f}x)")
print(f"Prefix hit : {pfx_stats['prefix_hit']}")

---
## 3. Continuous Batching vs Static Batching

**Static batching** groups N requests into fixed batches of size B.  
Every sequence in the batch must finish before the next batch can start.  
If one request needs 5 tokens and another needs 100, the GPU idles for 95 steps
on the short request's slot.

**Continuous batching** operates at *iteration* granularity:  
- Advance ALL in-flight sequences by one token per step.
- The instant a sequence hits EOS, its slot is filled with the next queued request.
- GPU never waits for laggards.

The benchmark uses a heterogeneous workload: some requests need 10 tokens,
others 50–60, to maximise the scheduling gap.

In [ ]:
from engine.continuous_batching import StaticBatchingEngine, ContinuousBatchingEngine
import time

BATCH_PROMPTS = [
    "Tell me a very short story about a cat.",
    "What is the capital of France?",
    "Write a haiku about autumn leaves.",
    "Explain quantum entanglement briefly.",
    "Describe the water cycle in one sentence.",
    "What are the primary colors?",
    "Write a limerick about a programmer.",
    "How does photosynthesis work?",
]
BATCH_MAX_TOKENS = [60, 15, 20, 40, 20, 10, 50, 35]

print(f'Workload: {len(BATCH_PROMPTS)} requests, output lengths {BATCH_MAX_TOKENS}')

# Static
static_eng = StaticBatchingEngine()
t0 = time.perf_counter()
s_out = static_eng.run(BATCH_PROMPTS, BATCH_MAX_TOKENS, batch_size=4)
s_wall = time.perf_counter() - t0

# Continuous
cont_eng = ContinuousBatchingEngine()
t0 = time.perf_counter()
c_out = cont_eng.run(BATCH_PROMPTS, BATCH_MAX_TOKENS, max_batch_size=4)
c_wall = time.perf_counter() - t0

print(f"\n{'Strategy':<28} | {'Time':>8} | {'Tok/s':>8} | {'Total tok':>10}")
print('-' * 62)
print(f"{'Static (B=4)':<28} | {s_out['total_time_s']:>7.2f}s | {s_out['throughput_tps']:>8.1f} | {s_out['total_tokens']:>10}")
print(f"{'Continuous (max=4)':<28} | {c_out['total_time_s']:>7.2f}s | {c_out['throughput_tps']:>8.1f} | {c_out['total_tokens']:>10}")
print(f"\nSpeedup: {s_out['total_time_s']/c_out['total_time_s']:.2f}x")

---
## 4. Speculative Decoding

**Idea:** run a small *draft* model to speculatively generate k tokens cheaply,
then verify all k in a single *target* model forward pass.

**Acceptance sampling** guarantees the output distribution is identical to
sampling from the target alone:

$$
\text{Accept } x \text{ with prob} = \min\!\left(1,\; \frac{P_{\text{target}}(x)}{P_{\text{draft}}(x)}\right)
$$

On rejection, resample from the *residual* $(P_T - P_D)^+$ normalised.

**Expected tokens per target call** = $1 + k \cdot \alpha$ where $\alpha$ = mean accept rate.

Here: draft = **gpt2** (117 M), target = **gpt2-medium** (345 M), k = 4.

In [ ]:
from engine.speculative import SpeculativeEngine
from engine.kv_cache import KVCacheEngine

# Target-only baseline (gpt2-medium + KV cache)
print('Loading target-only baseline ...')
target_eng = KVCacheEngine(model_name='gpt2-medium')
_, target_stats = target_eng.generate(PROMPT, max_new_tokens=N_TOKENS)

# Speculative
print('Loading speculative engine ...')
spec_eng = SpeculativeEngine()
_, spec_stats = spec_eng.generate(PROMPT, max_new_tokens=N_TOKENS, k=4)

print(f"\n{'Engine':<28} | {'TTFT':>8} | {'Tok/s':>8}")
print('-' * 50)
print(f"{'gpt2-medium (KV cache)':<28} | {target_stats['ttft_ms']:>7.1f}ms | {target_stats['throughput_tps']:>8.1f}")
print(f"{'Speculative (k=4)':<28} | {spec_stats['ttft_ms']:>7.1f}ms | {spec_stats['throughput_tps']:>8.1f}")
print(f"\nAccept rate       : {spec_stats['accept_rate']}")
print(f"Tokens/target call: {spec_stats['tok_per_target_call']}")
print(f"Speedup           : {spec_stats['throughput_tps']/target_stats['throughput_tps']:.2f}x")

---
## 5. Quantization

Reduce weight precision to save memory and accelerate matrix multiplications.

| Mode | Bits/weight | Notes |
|------|-------------|-------|
| FP32 | 32 | full precision baseline |
| INT8 | 8 | `torch.ao.quantization.quantize_dynamic`; ~2× smaller |
| W4A16 | 4 | custom groupwise quant; ~4× smaller; dequant to FP16 before matmul |

**W4A16** means weights are stored in 4 bits, activations stay in FP16.  
Production stacks (bitsandbytes, GPTQ, AWQ) use optimised CUDA kernels for
the dequant; our implementation dequantises on the fly to show the concept.

In [ ]:
from engine.quantization import QuantizationEngine

quant_results = {}
for mode in ('fp32', 'int8', 'w4'):
    q_eng = QuantizationEngine(mode=mode)
    _, stats = q_eng.generate(PROMPT, max_new_tokens=N_TOKENS)
    quant_results[mode] = stats
    print(f"{mode.upper():6s}: {stats['model_size_mb']:>8}  |  {stats['throughput_tps']:>6.1f} tok/s  |  TTFT {stats['ttft_ms']:.1f} ms")

baseline_size = quant_results['fp32']['model_size_mb']
print(f"\nINT8 compression: {baseline_size} → {quant_results['int8']['model_size_mb']}")
print(f"W4   compression: {baseline_size} → {quant_results['w4']['model_size_mb']}")

---
## Summary: before/after numbers

In [ ]:
from benchmarks.metrics import BenchmarkResult, print_table
import torch

def mem():
    return torch.cuda.max_memory_allocated() / 1e6 if torch.cuda.is_available() else 0

rows = [
    BenchmarkResult('Baseline (no KV cache)',       baseline_stats['ttft_ms'],  baseline_stats['throughput_tps'],  baseline_stats['latency_ms'],  mem()),
    BenchmarkResult('KV cache',                     kv_stats['ttft_ms'],        kv_stats['throughput_tps'],        kv_stats['latency_ms'],        mem()),
    BenchmarkResult('KV cache + prefix reuse',      pfx_stats['ttft_ms'],       pfx_stats['throughput_tps'],       pfx_stats['latency_ms'],       mem()),
    BenchmarkResult('Static batching',              0, s_out['throughput_tps'], s_out['total_time_s']*1000,       mem(), {'total_tok': s_out['total_tokens']}),
    BenchmarkResult('Continuous batching',          0, c_out['throughput_tps'], c_out['total_time_s']*1000,       mem(), {'total_tok': c_out['total_tokens']}),
    BenchmarkResult('gpt2-medium (KV cache)',       target_stats['ttft_ms'],    target_stats['throughput_tps'],    target_stats['latency_ms'],    mem()),
    BenchmarkResult('Speculative (k=4)',            spec_stats['ttft_ms'],      spec_stats['throughput_tps'],      spec_stats['latency_ms'],      mem(), {'accept': spec_stats['accept_rate']}),
    BenchmarkResult('FP32',                         quant_results['fp32']['ttft_ms'],  quant_results['fp32']['throughput_tps'],  quant_results['fp32']['latency_ms'],  mem(), {'size': quant_results['fp32']['model_size_mb']}),
    BenchmarkResult('INT8',                         quant_results['int8']['ttft_ms'],  quant_results['int8']['throughput_tps'],  quant_results['int8']['latency_ms'],  mem(), {'size': quant_results['int8']['model_size_mb']}),
    BenchmarkResult('W4A16',                        quant_results['w4']['ttft_ms'],    quant_results['w4']['throughput_tps'],    quant_results['w4']['latency_ms'],    mem(), {'size': quant_results['w4']['model_size_mb']}),
]
print_table(rows)